In [ ]:
import os
import django
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'krgate.settings')
django.setup()

from OBE.models import (
    AssessmentDraft, Review1Mark, Review2Mark, FinalInternalMark
)
from academics.models import TeachingAssignment, Subject
from OBE.services.final_internal_marks import _compute_tcpr_final_total

# Filter for TDC006 courses
tas = TeachingAssignment.objects.filter(subject__code__iexact='TDC006')
print(f"Found {tas.count()} assignments for TDC006")

for ta in tas:
    print(f"\nTA ID: {ta.id}")
    subject = ta.subject
    students = ta.section.studentsectionassignment_set.filter(end_date__isnull=True).values('student_id', 'student__reg_no')
    
    for s in students:
        sid = s['student_id']
        reg_no = s['student__reg_no']
        if reg_no != 'tests1': continue
        
        student_ref = {'id': sid, 'reg_no': reg_no}
        
        # Test direct compute
        res = _compute_tcpr_final_total(ta=ta, subject=subject, student=student_ref, ta_id=ta.id, return_details=True)
        print(f"Student {reg_no} (ID {sid}) Result:")
        import json
        print(json.dumps(res, indent=2))
        
        # Check stored FIM
        fim = FinalInternalMark.objects.filter(teaching_assignment_id=ta.id, student_id=sid).first()
        if fim:
            print(f"Stored Final Mark: {fim.final_mark}")
        else:
            print("No stored FinalInternalMark found")
